# Fraud Detection – Comprehensive Data Cleaning & EDA

**Project:** Fraud Detection for E-commerce and Bank Transactions  
**Company:** Adey Innovations Inc.  

## Business Objective
This project aims to build a robust fraud detection system for both e-commerce transactions (Fraud_Data.csv) and credit card transactions (creditcard.csv). The goal is to identify fraudulent patterns, engineer meaningful features, and prepare clean datasets for machine learning models.

## Datasets
1. **Fraud_Data.csv**: E-commerce transaction data with geolocation (IP-based)
2. **creditcard.csv**: Bank credit card transaction data
3. **IpAddress_to_Country.csv**: IP address to country mapping

## Workflow
- Data loading and inspection
- Data cleaning (missing values, duplicates, invalid data)
- Exploratory Data Analysis (univariate & bivariate)
- Class distribution analysis
- Feature engineering
- Data transformation and scaling
- Class imbalance handling
- Model-ready data preparation


In [ ]:
# DIAGNOSTIC CELL - Run this first to check your environment
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version_info)
print("\nChecking if packages are installed...")

try:
    import pandas as pd
    print("✓ pandas:", pd.__version__)
except ImportError:
    print("❌ pandas NOT FOUND")
    print("\n🔧 Installing packages...")
    import subprocess
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--user', '--break-system-packages', 
                            'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'imbalanced-learn'],
                          capture_output=True, text=True)
    print(result.stdout)
    if result.returncode == 0:
        print("✓ Packages installed! RESTART KERNEL NOW (Kernel → Restart)")
    else:
        print("❌ Installation failed:", result.stderr)

try:
    import numpy as np
    print("✓ numpy:", np.__version__)
except ImportError:
    print("❌ numpy NOT FOUND")

try:
    import sklearn
    print("✓ scikit-learn:", sklearn.__version__)
except ImportError:
    print("❌ scikit-learn NOT FOUND")


## 1. Import Required Libraries

In [ ]:
import sys
import os
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path to import src modules
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

# Import reusable functions from src modules (production-grade code)
from src.data_loading import load_data
from src.data_cleaning import clean_data, standardize_datatypes
from src.feature_engineering import engineer_features, merge_geolocation
from src.eda_utils import (
    analyze_class_distribution,
    analyze_missing_values,
    analyze_duplicates,
    get_data_summary
)
from src.preprocessing import (
    train_test_split_data,
    scale_features,
    encode_categorical_features,
    handle_class_imbalance
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('default')
sns.set_palette("husl")

print("✓ All libraries and modules imported successfully")
print("✓ Using reusable functions from src modules (production-grade code)")

## 2. Using Reusable Functions from src Modules

**Best Practice:** Instead of defining functions inline, we import reusable functions from the `src` modules. This ensures:
- ✅ Code reusability across notebooks and scripts
- ✅ Consistent error handling
- ✅ Easier testing and maintenance
- ✅ Production-grade code structure

All functions are imported from:
- `src.data_loading` - Load data with error handling
- `src.data_cleaning` - Clean and standardize data types
- `src.feature_engineering` - Engineer features and merge geolocation
- `src.eda_utils` - EDA utilities (class distribution, missing values, etc.)
- `src.preprocessing` - Preprocessing pipeline (scaling, encoding, SMOTE)

In [ ]:
# All functions are imported from src modules (see cell above)
# This ensures code reusability and production-grade practices

print("✓ Ready to use functions from src modules")
print("  Available functions:")
print("  - load_data()")
print("  - clean_data(), standardize_datatypes()")
print("  - engineer_features(), merge_geolocation()")
print("  - analyze_class_distribution(), analyze_missing_values(), analyze_duplicates(), get_data_summary()")
print("  - train_test_split_data(), scale_features(), encode_categorical_features(), handle_class_imbalance()")

## 3. Load All Datasets

In [ ]:
# Load datasets
fraud = load_data('../data/raw/Fraud_Data.csv')
ip = load_data('../data/raw/IpAddress_to_Country.csv')
creditcard = load_data('../data/raw/creditcard.csv')

## 4. Initial Data Inspection

### 4.1 Fraud_Data.csv (E-commerce Transactions)

In [ ]:
print("="*60)
print("FRAUD_DATA.CSV - E-COMMERCE TRANSACTIONS")
print("="*60)

# Comprehensive data summary using utility function
summary_fraud = get_data_summary(fraud, 'Fraud_Data')

print("\n📋 First Few Rows:")
display(fraud.head())

print("\n📈 Descriptive Statistics (Numeric Features):")
display(fraud.describe())

# Missing values analysis using utility function
missing_fraud = analyze_missing_values(fraud, 'Fraud_Data')

# Duplicate analysis using utility function
duplicates_fraud = analyze_duplicates(fraud, 'Fraud_Data')

# Data types analysis
print("\n📊 Data Types Summary:")
print(fraud.dtypes.value_counts())

# Check for target column
if 'class' in fraud.columns:
    print("\n✓ Target column 'class' found")
else:
    print("\n⚠ Warning: Target column 'class' not found!")

### 4.2 creditcard.csv (Bank Credit Card Transactions)

In [ ]:
print("="*60)
print("CREDITCARD.CSV - BANK CREDIT CARD TRANSACTIONS")
print("="*60)

# Comprehensive data summary using utility function
summary_cc = get_data_summary(creditcard, 'CreditCard')

print("\n📋 First Few Rows:")
display(creditcard.head())

print("\n📈 Descriptive Statistics (Numeric Features):")
display(creditcard.describe())

# Missing values analysis using utility function
missing_cc = analyze_missing_values(creditcard, 'CreditCard')

# Duplicate analysis using utility function
duplicates_cc = analyze_duplicates(creditcard, 'CreditCard')

# Data types analysis
print("\n📊 Data Types Summary:")
print(creditcard.dtypes.value_counts())

# Check for target column
if 'Class' in creditcard.columns:
    print("\n✓ Target column 'Class' found")
elif 'class' in creditcard.columns:
    print("\n✓ Target column 'class' found")
else:
    print("\n⚠ Warning: No target column found!")

## 5. Data Cleaning

### 5.1 Clean Fraud_Data.csv

In [ ]:
# Standardize data types using reusable function
fraud = standardize_datatypes(fraud, dataset_type='fraud_data')

# Clean data using reusable function
fraud = clean_data(
    fraud, 
    'Fraud_Data.csv',
    critical_cols=['user_id', 'purchase_time', 'class'],
    age_col='age',
    value_col='purchase_value'
)

### 5.2 Clean creditcard.csv

In [ ]:
# Standardize data types using reusable function
creditcard = standardize_datatypes(creditcard, dataset_type='creditcard')

# Rename 'Class' to 'class' for consistency (before cleaning)
if 'Class' in creditcard.columns and 'class' not in creditcard.columns:
    creditcard.rename(columns={'Class': 'class'}, inplace=True)

# Clean credit card data using reusable function
creditcard = clean_data(
    creditcard,
    'creditcard.csv',
    critical_cols=['class'] if 'class' in creditcard.columns else None,
    value_col='Amount' if 'Amount' in creditcard.columns else None
)

### 5.3 Clean IP Address Data

In [ ]:
ip = ip.drop_duplicates()
ip['lower_bound_ip_address'] = ip['lower_bound_ip_address'].astype(int)
ip['upper_bound_ip_address'] = ip['upper_bound_ip_address'].astype(int)
print("✓ IP address data cleaned")

## 6. Class Distribution Analysis

Understanding class imbalance is critical for fraud detection models.

In [ ]:
analyze_class_distribution(fraud, 'class', 'Fraud_Data (E-commerce)')

In [ ]:
analyze_class_distribution(creditcard, 'class', 'CreditCard (Bank Transactions)')

## 7. Exploratory Data Analysis (EDA)

### 7.1 Univariate Analysis - Fraud_Data

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Purchase value distribution
fraud['purchase_value'].hist(bins=50, ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('Purchase Value Distribution')
axes[0,0].set_xlabel('Purchase Value')
axes[0,0].set_ylabel('Frequency')

# Age distribution
fraud['age'].hist(bins=30, ax=axes[0,1], edgecolor='black', color='orange')
axes[0,1].set_title('Age Distribution')
axes[0,1].set_xlabel('Age')
axes[0,1].set_ylabel('Frequency')

# Source distribution
fraud['source'].value_counts().plot(kind='bar', ax=axes[1,0], color='green')
axes[1,0].set_title('Traffic Source Distribution')
axes[1,0].set_xlabel('Source')
axes[1,0].tick_params(axis='x', rotation=45)

# Browser distribution
fraud['browser'].value_counts().plot(kind='bar', ax=axes[1,1], color='purple')
axes[1,1].set_title('Browser Distribution')
axes[1,1].set_xlabel('Browser')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 7.2 Univariate Analysis - CreditCard

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount distribution
if 'Amount' in creditcard.columns:
    creditcard['Amount'].hist(bins=50, ax=axes[0], edgecolor='black')
    axes[0].set_title('Transaction Amount Distribution')
    axes[0].set_xlabel('Amount')
    axes[0].set_ylabel('Frequency')
    axes[0].set_yscale('log')  # Log scale due to skewness
    
    # Log-transformed amount
    np.log1p(creditcard['Amount']).hist(bins=50, ax=axes[1], edgecolor='black', color='orange')
    axes[1].set_title('Log(Amount) Distribution')
    axes[1].set_xlabel('Log(Amount)')
    axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### 7.3 Bivariate Analysis - Fraud vs Features

In [ ]:
# Fraud_Data bivariate analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='class', y='purchase_value', data=fraud, ax=axes[0])
axes[0].set_title('Purchase Value vs Fraud')
axes[0].set_xlabel('Class (0=Legitimate, 1=Fraud)')

sns.boxplot(x='class', y='age', data=fraud, ax=axes[1])
axes[1].set_title('Age vs Fraud')
axes[1].set_xlabel('Class (0=Legitimate, 1=Fraud)')

plt.tight_layout()
plt.show()

### 7.5 Correlation Analysis

Correlation analysis helps identify relationships between features and the target variable.


In [ ]:
# Correlation analysis for Fraud_Data
print("="*60)
print("CORRELATION ANALYSIS - FRAUD_DATA")
print("="*60)

# Select numeric columns
fraud_numeric_cols = fraud.select_dtypes(include=[np.number]).columns.tolist()
if 'class' in fraud_numeric_cols:
    fraud_corr = fraud[fraud_numeric_cols].corr()['class'].sort_values(ascending=False)
    
    print("\n📊 Correlation with Target (class):")
    print(fraud_corr)
    
    # Visualize correlation
    plt.figure(figsize=(10, 8))
    fraud_corr_abs = fraud_corr.abs().sort_values(ascending=False)
    fraud_corr_abs.drop('class', inplace=True)  # Remove self-correlation
    fraud_corr_abs.plot(kind='barh')
    plt.title('Feature Correlation with Fraud Target (Fraud_Data)', fontweight='bold')
    plt.xlabel('Absolute Correlation')
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Insights:")
    print("  → Higher absolute correlation indicates stronger relationship with fraud")
    print("  → Positive correlation: feature increases with fraud")
    print("  → Negative correlation: feature decreases with fraud")


In [ ]:
# Correlation analysis for CreditCard
print("="*60)
print("CORRELATION ANALYSIS - CREDITCARD")
print("="*60)

# Select numeric columns
cc_numeric_cols = creditcard.select_dtypes(include=[np.number]).columns.tolist()
if 'class' in cc_numeric_cols or 'Class' in cc_numeric_cols:
    target_col = 'class' if 'class' in cc_numeric_cols else 'Class'
    cc_corr = creditcard[cc_numeric_cols].corr()[target_col].sort_values(ascending=False)
    
    print(f"\n📊 Correlation with Target ({target_col}):")
    print(f"\nTop 10 features most correlated with fraud:")
    print(cc_corr.head(11))  # Include target itself
    
    # Visualize correlation heatmap for top features
    top_features = cc_corr.abs().sort_values(ascending=False).head(11).index.tolist()
    top_features.remove(target_col)  # Remove target from features list
    
    plt.figure(figsize=(12, 8))
    corr_matrix = creditcard[top_features + [target_col]].corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Heatmap: Top Features vs Fraud (CreditCard)', fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Insights:")
    print("  → V-features show varying correlations (some strong, some weak)")
    print("  → Amount and Time may show different patterns for fraud")
    print("  → Strong correlations can guide feature selection")


### 7.4 Enhanced Bivariate Analysis - CreditCard (Comprehensive)

Comprehensive analysis showing relationships between features and fraud target.


In [ ]:
# Enhanced bivariate analysis for CreditCard
# Ensure 'class' column exists (rename if needed)
if 'Class' in creditcard.columns and 'class' not in creditcard.columns:
    creditcard['class'] = creditcard['Class']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Amount vs Fraud
if 'Amount' in creditcard.columns:
    sns.boxplot(x='class', y='Amount', data=creditcard, ax=axes[0,0])
    axes[0,0].set_title('Transaction Amount vs Fraud', fontsize=12, fontweight='bold')
    axes[0,0].set_xlabel('Class (0=Legitimate, 1=Fraud)')
    axes[0,0].set_ylabel('Amount')
    axes[0,0].set_yscale('log')
    
    # Violin plot for better distribution view
    sns.violinplot(x='class', y='Amount', data=creditcard, ax=axes[0,1])
    axes[0,1].set_title('Amount Distribution by Class (Violin Plot)', fontsize=12, fontweight='bold')
    axes[0,1].set_xlabel('Class (0=Legitimate, 1=Fraud)')
    axes[0,1].set_ylabel('Amount')
    axes[0,1].set_yscale('log')

# Time vs Fraud
if 'Time' in creditcard.columns:
    sns.boxplot(x='class', y='Time', data=creditcard, ax=axes[1,0])
    axes[1,0].set_title('Time vs Fraud', fontsize=12, fontweight='bold')
    axes[1,0].set_xlabel('Class (0=Legitimate, 1=Fraud)')
    axes[1,0].set_ylabel('Time (seconds)')

# Sample V-feature vs Fraud (V1 as example)
v_cols = [col for col in creditcard.columns if col.startswith('V')]
if v_cols:
    sns.boxplot(x='class', y=v_cols[0], data=creditcard, ax=axes[1,1])
    axes[1,1].set_title(f'{v_cols[0]} vs Fraud (Sample PCA Feature)', fontsize=12, fontweight='bold')
    axes[1,1].set_xlabel('Class (0=Legitimate, 1=Fraud)')
    axes[1,1].set_ylabel(v_cols[0])

plt.tight_layout()
plt.show()

# Statistical comparison
print("\n📊 Statistical Comparison by Class:")
if 'Amount' in creditcard.columns:
    print("\nAmount Statistics by Class:")
    print(creditcard.groupby('class')['Amount'].describe())

if 'Time' in creditcard.columns:
    print("\nTime Statistics by Class:")
    print(creditcard.groupby('class')['Time'].describe())


### 8.1 Enhanced Geolocation Visualization

Visualizing fraud patterns by country helps identify high-risk regions.


In [ ]:
# Enhanced visualization of fraud by country
if 'country' in fraud.columns:
    fraud_by_country = fraud.groupby('country')['class'].agg(['mean', 'count']).sort_values('mean', ascending=False)
    fraud_by_country.columns = ['fraud_rate', 'transaction_count']
    fraud_by_country = fraud_by_country[fraud_by_country['transaction_count'] >= 10]  # At least 10 transactions
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 12))
    
    # Top 15 countries by fraud rate
    top_countries = fraud_by_country.head(15)
    axes[0].barh(range(len(top_countries)), top_countries['fraud_rate'], color='red', alpha=0.7)
    axes[0].set_yticks(range(len(top_countries)))
    axes[0].set_yticklabels(top_countries.index)
    axes[0].set_xlabel('Fraud Rate', fontweight='bold')
    axes[0].set_title('Top 15 Countries by Fraud Rate', fontweight='bold', fontsize=14)
    axes[0].invert_yaxis()
    axes[0].grid(axis='x', alpha=0.3)
    
    # Add fraud rate percentages as text
    for i, (country, row) in enumerate(top_countries.iterrows()):
        axes[0].text(row['fraud_rate'] + 0.01, i, f"{row['fraud_rate']*100:.1f}%", 
                    va='center', fontweight='bold')
    
    # Top 15 countries by transaction volume with fraud rate
    top_volume = fraud_by_country.nlargest(15, 'transaction_count')
    axes[1].barh(range(len(top_volume)), top_volume['fraud_rate'], color='orange', alpha=0.7)
    axes[1].set_yticks(range(len(top_volume)))
    axes[1].set_yticklabels(top_volume.index)
    axes[1].set_xlabel('Fraud Rate', fontweight='bold')
    axes[1].set_title('Top 15 Countries by Transaction Volume (with Fraud Rate)', fontweight='bold', fontsize=14)
    axes[1].invert_yaxis()
    axes[1].grid(axis='x', alpha=0.3)
    
    # Add transaction counts as text
    for i, (country, row) in enumerate(top_volume.iterrows()):
        axes[1].text(row['fraud_rate'] + 0.01, i, f"{int(row['transaction_count']):,} txns", 
                    va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Key Insights:")
    print("  → Countries with high fraud rates may indicate geographic fraud patterns")
    print("  → High-volume countries with elevated fraud rates require special attention")
    print("  → This information can be used to create country-based risk features")


In [ ]:
# CreditCard bivariate analysis
if 'Amount' in creditcard.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x='class', y='Amount', data=creditcard)
    plt.title('Transaction Amount vs Fraud (CreditCard)')
    plt.xlabel('Class (0=Legitimate, 1=Fraud)')
    plt.yscale('log')
    plt.show()

## 8. Geolocation Integration (Fraud_Data only)

In [ ]:
fraud = merge_geolocation(fraud, ip)

# Analyze fraud rate by country
fraud_by_country = fraud.groupby('country')['class'].agg(['mean', 'count']).sort_values('mean', ascending=False)
fraud_by_country = fraud_by_country[fraud_by_country['count'] >= 10]  # At least 10 transactions

print("\n📍 Top 10 Countries by Fraud Rate:")
print(fraud_by_country.head(10))

## 9. Feature Engineering

In [ ]:
fraud = engineer_features(fraud, dataset_type='fraud_data')

print("\n✓ Engineered features for Fraud_Data:")
new_cols = ['hour_of_day', 'day_of_week', 'is_weekend', 'is_night', 
           'time_since_signup', 'quick_purchase', 'rapid_transactions']
print(fraud[new_cols].head())

In [ ]:
creditcard = engineer_features(creditcard, dataset_type='creditcard')

print("\n✓ Engineered features for CreditCard")
if 'features_mean' in creditcard.columns:
    print(creditcard[['features_mean', 'features_std', 'log_amount']].head())

**Note:** The above cell uses an inline function for demonstration. Below we show the proper approach using src modules.


In [ ]:
# Proper approach: Use reusable functions from src modules
# Prepare Fraud_Data for modeling
fraud_numeric = fraud.select_dtypes(include=[np.number]).copy()
fraud_numeric['class'] = fraud['class']

# Train-test split using reusable function (prevents data leakage)
X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = train_test_split_data(
    fraud_numeric, target_col='class', test_size=0.2, random_state=42
)

# Also prepare CreditCard data
creditcard_numeric = creditcard.select_dtypes(include=[np.number]).copy()
if 'class' not in creditcard_numeric.columns and 'class' in creditcard.columns:
    creditcard_numeric['class'] = creditcard['class']

X_train_cc, X_test_cc, y_train_cc, y_test_cc = train_test_split_data(
    creditcard_numeric, target_col='class', test_size=0.2, random_state=42
)


## 10. Data Transformation & Preprocessing Pipeline

### 10.1 Train-Test Split (Preventing Data Leakage)

In [ ]:
def prepare_modeling_data(df, dataset_name):
    """
    Prepare data for modeling with proper train-test split.
    """
    print(f"\n{'='*60}")
    print(f"Preparing {dataset_name} for Modeling")
    print(f"={'*60}")
    
    # Separate features and target
    X = df.drop('class', axis=1)
    y = df['class']
    
    # Train-test split (80-20)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"✓ Train set: {X_train.shape[0]} samples")
    print(f"✓ Test set: {X_test.shape[0]} samples")
    print(f"\nTrain class distribution:")
    print(y_train.value_counts())
    
    return X_train, X_test, y_train, y_test

# For demonstration, we'll prepare Fraud_Data
# Select numeric features only for now
fraud_numeric = fraud.select_dtypes(include=[np.number]).copy()
fraud_numeric['class'] = fraud['class']

X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = prepare_modeling_data(
    fraud_numeric, 'Fraud_Data'
)

### 10.2 Feature Scaling (StandardScaler)

### 10.4 Complete Preprocessing Pipeline (Using src Modules)

Complete pipeline demonstration with proper class distribution documentation.


In [ ]:
# Complete preprocessing pipeline for Fraud_Data using src modules
print("="*80)
print("COMPLETE PREPROCESSING PIPELINE - FRAUD_DATA")
print("="*80)

# Step 1: Train-test split (already done above, but showing for completeness)
print("\n[Step 1] Train-Test Split:")
print(f"  Train: {X_train_fraud.shape[0]} samples")
print(f"  Test: {X_test_fraud.shape[0]} samples")

# Step 2: Feature scaling
print("\n[Step 2] Feature Scaling:")
X_train_fraud_scaled, X_test_fraud_scaled, scaler_fraud = scale_features(
    X_train_fraud, X_test_fraud, scaler_type='standard'
)

# Step 3: Class imbalance handling (SMOTE) - ONLY on training data
print("\n[Step 3] Class Imbalance Handling (SMOTE):")
print("\n📊 Class Distribution BEFORE SMOTE:")
print(y_train_fraud.value_counts().sort_index())
print(f"  Imbalance Ratio: {y_train_fraud.value_counts().max() / y_train_fraud.value_counts().min():.2f}:1")

X_train_fraud_smote, y_train_fraud_smote = handle_class_imbalance(
    X_train_fraud_scaled, y_train_fraud, method='smote', random_state=42
)

print("\n📊 Class Distribution AFTER SMOTE:")
print(pd.Series(y_train_fraud_smote).value_counts().sort_index())

# Visualize before/after
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
y_train_fraud.value_counts().sort_index().plot(kind='bar', ax=ax1, color=['green', 'red'])
ax1.set_title('Before SMOTE', fontweight='bold')
ax1.set_xlabel('Class')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=0)

pd.Series(y_train_fraud_smote).value_counts().sort_index().plot(kind='bar', ax=ax2, color=['green', 'red'])
ax2.set_title('After SMOTE', fontweight='bold')
ax2.set_xlabel('Class')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("\n✓ Complete preprocessing pipeline finished for Fraud_Data")
print("  → Training data: Balanced (SMOTE applied)")
print("  → Test data: Imbalanced (real-world distribution preserved)")


In [ ]:
# Complete preprocessing pipeline for CreditCard using src modules
print("="*80)
print("COMPLETE PREPROCESSING PIPELINE - CREDITCARD")
print("="*80)

# Step 1: Train-test split
print("\n[Step 1] Train-Test Split:")
print(f"  Train: {X_train_cc.shape[0]} samples")
print(f"  Test: {X_test_cc.shape[0]} samples")

# Step 2: Feature scaling
print("\n[Step 2] Feature Scaling:")
X_train_cc_scaled, X_test_cc_scaled, scaler_cc = scale_features(
    X_train_cc, X_test_cc, scaler_type='standard'
)

# Step 3: Class imbalance handling (SMOTE) - ONLY on training data
print("\n[Step 3] Class Imbalance Handling (SMOTE):")
print("\n📊 Class Distribution BEFORE SMOTE:")
print(y_train_cc.value_counts().sort_index())
if len(y_train_cc.value_counts()) == 2:
    print(f"  Imbalance Ratio: {y_train_cc.value_counts().max() / y_train_cc.value_counts().min():.2f}:1")
    print("  ⚠ EXTREME IMBALANCE - SMOTE critical!")

X_train_cc_smote, y_train_cc_smote = handle_class_imbalance(
    X_train_cc_scaled, y_train_cc, method='smote', random_state=42
)

print("\n📊 Class Distribution AFTER SMOTE:")
print(pd.Series(y_train_cc_smote).value_counts().sort_index())

# Visualize before/after
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
y_train_cc.value_counts().sort_index().plot(kind='bar', ax=ax1, color=['green', 'red'])
ax1.set_title('Before SMOTE', fontweight='bold')
ax1.set_xlabel('Class')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=0)

pd.Series(y_train_cc_smote).value_counts().sort_index().plot(kind='bar', ax=ax2, color=['green', 'red'])
ax2.set_title('After SMOTE', fontweight='bold')
ax2.set_xlabel('Class')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("\n✓ Complete preprocessing pipeline finished for CreditCard")
print("  → Training data: Balanced (SMOTE applied)")
print("  → Test data: Imbalanced (real-world distribution preserved)")


## 12. Save Processed Data

Save cleaned datasets and preprocessed train/test splits for modeling.


In [ ]:
import os

# Create processed directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save cleaned datasets
print("="*80)
print("SAVING PROCESSED DATA")
print("="*80)

# Save cleaned Fraud_Data
fraud.to_csv('../data/processed/fraud_data_cleaned.csv', index=False)
print("✓ Saved: fraud_data_cleaned.csv")

# Save cleaned CreditCard data (consistent with Fraud_Data)
creditcard.to_csv('../data/processed/creditcard_cleaned.csv', index=False)
print("✓ Saved: creditcard_cleaned.csv")

# Save Fraud_Data train-test splits with SMOTE
if 'X_train_fraud_smote' in globals() and 'y_train_fraud_smote' in globals():
    pd.DataFrame(X_train_fraud_smote, columns=X_train_fraud_scaled.columns).to_csv(
        '../data/processed/fraud_X_train_smote.csv', index=False
    )
    pd.Series(y_train_fraud_smote).to_csv(
        '../data/processed/fraud_y_train_smote.csv', index=False, header=['class']
    )
    X_test_fraud_scaled.to_csv('../data/processed/fraud_X_test.csv', index=False)
    y_test_fraud.to_csv('../data/processed/fraud_y_test.csv', index=False, header=['class'])
    print("\n✓ Saved Fraud_Data preprocessed sets:")
    print("  - fraud_X_train_smote.csv (SMOTE-balanced)")
    print("  - fraud_y_train_smote.csv")
    print("  - fraud_X_test.csv (imbalanced - real distribution)")
    print("  - fraud_y_test.csv")

# Save CreditCard train-test splits with SMOTE
if 'X_train_cc_smote' in globals() and 'y_train_cc_smote' in globals():
    pd.DataFrame(X_train_cc_smote, columns=X_train_cc_scaled.columns).to_csv(
        '../data/processed/creditcard_X_train_smote.csv', index=False
    )
    pd.Series(y_train_cc_smote).to_csv(
        '../data/processed/creditcard_y_train_smote.csv', index=False, header=['class']
    )
    X_test_cc_scaled.to_csv('../data/processed/creditcard_X_test.csv', index=False)
    y_test_cc.to_csv('../data/processed/creditcard_y_test.csv', index=False, header=['class'])
    print("\n✓ Saved CreditCard preprocessed sets:")
    print("  - creditcard_X_train_smote.csv (SMOTE-balanced)")
    print("  - creditcard_y_train_smote.csv")
    print("  - creditcard_X_test.csv (imbalanced - real distribution)")
    print("  - creditcard_y_test.csv")

print("\n" + "="*80)
print("DATA PREPROCESSING COMPLETE!")
print("="*80)
print("\n📋 Summary:")
print("  ✓ Both datasets cleaned and saved")
print("  ✓ Train/test splits created (80/20)")
print("  ✓ Features scaled (StandardScaler)")
print("  ✓ Class imbalance handled (SMOTE on training data only)")
print("  ✓ Test sets preserved with real-world distribution")
print("\n🎯 Next Steps:")
print("  1. Use cleaned datasets for further analysis")
print("  2. Use SMOTE-balanced train sets for model training")
print("  3. Evaluate models on imbalanced test sets (real-world scenario)")
print("  4. Consider additional feature engineering based on EDA insights")


In [ ]:
# Initialize scaler
scaler = StandardScaler()

# Fit on training data only (prevent leakage)
X_train_fraud_scaled = scaler.fit_transform(X_train_fraud)
X_test_fraud_scaled = scaler.transform(X_test_fraud)

# Convert back to DataFrame for readability
X_train_fraud_scaled = pd.DataFrame(
    X_train_fraud_scaled, 
    columns=X_train_fraud.columns,
    index=X_train_fraud.index
)
X_test_fraud_scaled = pd.DataFrame(
    X_test_fraud_scaled, 
    columns=X_test_fraud.columns,
    index=X_test_fraud.index
)

print("✓ Features scaled using StandardScaler")
print("\nScaled feature statistics (first 5 features):")
print(X_train_fraud_scaled.iloc[:, :5].describe())

### 10.3 Categorical Encoding (if needed)

In [ ]:
# Example: One-Hot Encoding for categorical features
categorical_cols = ['source', 'browser', 'sex']

# This is demonstration code - in practice, fit on training set only
fraud_encoded = pd.get_dummies(fraud, columns=categorical_cols, drop_first=True)

print(f"✓ Categorical encoding complete")
print(f"Original columns: {fraud.shape[1]}")
print(f"After encoding: {fraud_encoded.shape[1]}")
print(f"\nNew columns created:")
new_cols_enc = [col for col in fraud_encoded.columns if col not in fraud.columns]
print(new_cols_enc[:10])  # Show first 10

## 11. Class Imbalance Handling

**Critical Note:** Apply only to TRAINING data to prevent data leakage!

### 11.1 SMOTE (Synthetic Minority Over-sampling)

In [ ]:
print("="*60)
print("SMOTE - Synthetic Minority Over-sampling")
print("="*60)

print("\n📊 Before SMOTE:")
print(y_train_fraud.value_counts())

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_fraud_scaled, y_train_fraud)

print("\n📊 After SMOTE:")
print(pd.Series(y_train_smote).value_counts())

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

y_train_fraud.value_counts().plot(kind='bar', ax=ax1, color=['green', 'red'])
ax1.set_title('Before SMOTE')
ax1.set_xlabel('Class')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=0)

pd.Series(y_train_smote).value_counts().plot(kind='bar', ax=ax2, color=['green', 'red'])
ax2.set_title('After SMOTE')
ax2.set_xlabel('Class')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("\n✓ SMOTE applied successfully - Class balance achieved!")

### 11.2 Random Under-sampling (Alternative)

In [ ]:
print("="*60)
print("Random Under-sampling (Alternative Approach)")
print("="*60)

print("\n📊 Before Under-sampling:")
print(y_train_fraud.value_counts())

# Apply Random Under-sampling
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train_fraud_scaled, y_train_fraud)

print("\n📊 After Under-sampling:")
print(pd.Series(y_train_under).value_counts())

print(f"\n⚠ Note: Under-sampling reduces dataset size from {len(y_train_fraud)} to {len(y_train_under)}")
print("   → Use with caution as it discards majority class samples")

## 12. Save Cleaned Data

Save processed datasets for modeling phase.

In [ ]:
# Save cleaned Fraud_Data
fraud.to_csv('../data/processed/fraud_data_cleaned.csv', index=False)
print("✓ Saved: fraud_data_cleaned.csv")

# Save cleaned CreditCard data
creditcard.to_csv('../data/processed/creditcard_cleaned.csv', index=False)
print("✓ Saved: creditcard_cleaned.csv")

# Save train-test splits with SMOTE
pd.DataFrame(X_train_smote, columns=X_train_fraud_scaled.columns).to_csv(
    '../data/processed/fraud_X_train_smote.csv', index=False
)
pd.Series(y_train_smote).to_csv(
    '../data/processed/fraud_y_train_smote.csv', index=False, header=['class']
)

X_test_fraud_scaled.to_csv('../data/processed/fraud_X_test.csv', index=False)
y_test_fraud.to_csv('../data/processed/fraud_y_test.csv', index=False, header=['class'])

print("✓ Saved: SMOTE-balanced train/test sets")
print("\n" + "="*60)
print("DATA PREPROCESSING COMPLETE!")
print("="*60)
print("\nNext Steps:")
print("1. Use fraud_data_cleaned.csv for e-commerce fraud modeling")
print("2. Use creditcard_cleaned.csv for bank transaction fraud modeling")
print("3. Use SMOTE-balanced sets for training machine learning models")
print("4. Remember: Test set is left imbalanced (real-world distribution)")

## 13. Summary & Key Findings

### Data Quality
- Both datasets have been thoroughly cleaned and validated
- Missing values handled appropriately
- Duplicates removed
- Invalid data entries filtered out

### Class Imbalance
- **Fraud_Data**: Severe imbalance (~9% fraud)
- **CreditCard**: Extreme imbalance (check distribution above)
- **Solution Applied**: SMOTE for training data only

### Feature Engineering
- Time-based features created (hour, day, weekend, night)
- User behavior features (signup velocity, rapid transactions)
- Statistical aggregations for credit card data

### Data Transformation
- StandardScaler applied to numeric features
- Categorical variables encoded (one-hot encoding)
- Train-test split with stratification (80-20)

### Best Practices Implemented
✓ Reusable functions with error handling  
✓ Proper train-test splitting to prevent leakage  
✓ Scaling fitted on training data only  
✓ SMOTE applied to training set only  
✓ Comprehensive documentation and logging  